# Детальное описание функционала TrackAI

По каждому функционалу: задача, строки кода, изменённые файлы, артефакт на выходе.

## Сводная таблица

In [ ]:
import pandas as pd

summary = pd.DataFrame({
    '№': range(1, 18),
    'Функционал': [
        'Загрузка видео',
        'Анализ по ID',
        'Video Tracker (SLAM)',
        'Библиотека видео',
        'TrajectoryMap',
        'TrajectoryAnalysis Page',
        'TrajectoryAnalysis Component',
        'Pathfinding',
        'Точка отсчёта и направление',
        'Компас и калибровка',
        'API-клиент',
        'DWG → PNG',
        'PDF → PNG',
        'Библиотека планов',
        'Редактор плана',
        'Чат поддержки',
        'Telegram ошибки'
    ],
    'Строк': ['~390', '~630', '~2340', '~870', '3333', '2868', '2883', '642', '~360', '~450', '1155', '~540', '~195', '~750', '873', '~1350', '~240'],
    'Файлов': [2, 3, 4, 2, 1, 1, 1, 1, 1, 1, 1, 3, 2, 3, 1, 3, 1],
    'Артефакт': [
        'video_id, файл',
        'JSON траектории, PNG',
        'analysis.json, trajectory.png',
        'UI списка, траектория',
        'Визуализация на карте',
        'UI страницы',
        'Карточка анализа',
        'Скорректированные точки',
        'referencePoint, directionPoint',
        'Угол, масштаб',
        'ApiClient',
        'PNG план',
        'PNG план',
        'CRUD планов',
        'drawnPlan',
        'UI чата',
        'Уведомление'
    ]
})
summary

: 

---

## 1. Загрузка видео (Upload Video)

| Параметр | Значение |
|----------|----------|
| **Задача** | Загрузить видеофайл на сервер потоково (чанками 1 MB) без немедленного анализа. Поддержка больших файлов (до 500 MB). |
| **Строк кода** | Backend: ~150 (main.py) + Frontend: ~240 (api.ts) = **~390** |
| **Изменённые файлы** | `backend/main.py`, `src/lib/api.ts` |
| **Артефакт на выходе** | `{ success, video_id, filename, original_filename, file_size }` — идентификатор для последующего анализа; файл в `videos/` |

---

## 2. Анализ видео по ID (Analyze Video by ID)

| Параметр | Значение |
|----------|----------|
| **Задача** | Запустить анализ уже загруженного видео по `video_id`. Двухшаговый процесс: сначала upload, затем analyze. Фоновая обработка. |
| **Строк кода** | Backend: ~180 (main.py) + Frontend: ~150 (api.ts) + TrajectoryAnalysis: ~300 = **~630** |
| **Изменённые файлы** | `backend/main.py`, `src/lib/api.ts`, `src/pages/TrajectoryAnalysis.tsx` |
| **Артефакт на выходе** | `{ success, video_id, status: "queued" }`; после обработки — JSON с траекторией, turn_points, статистикой; PNG-визуализация в `outputs/` |

---

## 3. Обработка видео (Video Tracker — SLAM, стабилизация)

| Параметр | Значение |
|----------|----------|
| **Задача** | Извлечь траекторию движения из видео: Visual Odometry (SLAM), опциональная стабилизация, детекция поворотов, расчёт дистанции. |
| **Строк кода** | processor.py: **1539**, slam_wrapper.py: **456**, stabilization.py: **216**, video_utils.py: **123** = **~2340** |
| **Изменённые файлы** | `backend/video_tracker/src/processor.py`, `slam_wrapper.py`, `stabilization.py`, `utils/video_utils.py` |
| **Артефакт на выходе** | `*_analysis.json` (траектория, turn_points, video_info, processing_stats); `*_trajectory_square.png` — визуализация пути |

---

## 4. Библиотека видео (Video Library)

| Параметр | Значение |
|----------|----------|
| **Задача** | Список загруженных видео, загрузка сохранённого анализа по `video_id`, отображение траектории на карте. |
| **Строк кода** | Backend: ~240 (main.py: GET /videos, GET /video/{id}) + Frontend: **624** (VideoLibrary.tsx) = **~870** |
| **Изменённые файлы** | `backend/main.py`, `src/components/VideoLibrary.tsx` |
| **Артефакт на выходе** | UI: список видео с датой, размером, статусом; при «Загрузить анализ» — траектория на карте |

---

## 5. Карта траекторий (TrajectoryMap)

| Параметр | Значение |
|----------|----------|
| **Задача** | Интерактивная SVG-карта: план помещения, траектории, точка отсчёта, направление, компас, zoom/pan, pathfinding, калибровка масштаба. |
| **Строк кода** | **3 333** |
| **Изменённые файлы** | `src/components/TrajectoryMap.tsx` |
| **Артефакт на выходе** | Визуализация траекторий на плане с поворотом по направлению; компас; heatmap; экспорт |

---

## 6. Страница анализа траектории (TrajectoryAnalysis Page)

| Параметр | Значение |
|----------|----------|
| **Задача** | Главная страница: загрузка плана (img/PDF/DWG), установка точки отсчёта и направления, загрузка видео, запуск анализа, отображение результатов. |
| **Строк кода** | **2 868** |
| **Изменённые файлы** | `src/pages/TrajectoryAnalysis.tsx` |
| **Артефакт на выходе** | UI: план с маркерами, вкладки (План / Видео / Результаты), кнопки «Указать направление», «Сбросить точку» |

---

## 7. Компонент карточки анализа (TrajectoryAnalysis Component)

| Параметр | Значение |
|----------|----------|
| **Задача** | Карточка с вкладками: траектория, статистика, точки поворота. Интеграция TrajectoryMap, VideoLibrary. |
| **Строк кода** | **2 883** |
| **Изменённые файлы** | `src/components/TrajectoryAnalysis.tsx` |
| **Артефакт на выходе** | UI: карточка с графиком траектории, статистикой (дистанция, повороты), списком turn points |

---

## 8. Pathfinding по плану

| Параметр | Значение |
|----------|----------|
| **Задача** | Траектория следует плану, не проходит сквозь стены. A* на occupancy grid из изображения (luminance). |
| **Строк кода** | **642** |
| **Изменённые файлы** | `src/lib/pathfinding.ts` |
| **Артефакт на выходе** | Скорректированный массив точек `{x, y}[]`, обходящий стены на плане |

---

## 9. Точка отсчёта и направление (Reference Point & Direction)

| Параметр | Значение |
|----------|----------|
| **Задача** | Клик по плану → точка отсчёта; второй клик → направление. Коррекция координат при object-contain (img/SVG). |
| **Строк кода** | ~360 (handleFloorPlanClick, planImgLayout, overlay) в TrajectoryAnalysis page |
| **Изменённые файлы** | `src/pages/TrajectoryAnalysis.tsx` |
| **Артефакт на выходе** | `referencePoint`, `directionPoint` в localStorage; стрелка на плане; угол для поворота траектории |

---

## 10. Компас и калибровка направления

| Параметр | Значение |
|----------|----------|
| **Задача** | Угол поворота плана (0–360°), калибровка масштаба по двум точкам (известное расстояние в метрах). |
| **Строк кода** | ~450 в TrajectoryMap (compassAngle, scaleCalibStep, scalePointA/B) |
| **Изменённые файлы** | `src/components/TrajectoryMap.tsx` |
| **Артефакт на выходе** | Визуализация с повёрнутым планом; компас внизу; сохранение compassAngle в localStorage |

---

## 11. API-клиент (api.ts)

| Параметр | Значение |
|----------|----------|
| **Задача** | Единая точка доступа к backend: uploadVideo, analyzeVideoById, getVideos, getVideoAnalysis, plans CRUD, convertDwg, convertPdf. |
| **Строк кода** | **1 155** |
| **Изменённые файлы** | `src/lib/api.ts` |
| **Артефакт на выходе** | Класс ApiClient с методами; типы VideoAnalysisResult, VideoListItem, Plan |

---

## 12. Конвертация DWG → PNG

| Параметр | Значение |
|----------|----------|
| **Задача** | Загрузить DWG, конвертировать через ODA File Converter (DWG→DXF→SVG→PNG), вернуть PNG в base64. Асинхронно с polling статуса. |
| **Строк кода** | Backend: ~360 (main.py) + Frontend: ~180 (api.ts, TrajectoryAnalysis) = **~540** |
| **Изменённые файлы** | `backend/main.py`, `src/lib/api.ts`, `src/pages/TrajectoryAnalysis.tsx` |
| **Артефакт на выходе** | `{ job_id }` → `{ status, progress, png?, filename? }`; PNG-изображение плана |

---

## 13. Конвертация PDF → PNG

| Параметр | Значение |
|----------|----------|
| **Задача** | Первая страница PDF → PNG через pdftoppm (poppler-utils). |
| **Строк кода** | Backend: ~105 (main.py) + Frontend: ~90 = **~195** |
| **Изменённые файлы** | `backend/main.py`, `src/pages/TrajectoryAnalysis.tsx` |
| **Артефакт на выходе** | `{ success, png, filename }` — base64 PNG |

---

## 14. Библиотека планов (Plans CRUD)

| Параметр | Значение |
|----------|----------|
| **Задача** | Сохранение, список, удаление нарисованных планов. SQLite. |
| **Строк кода** | Backend: ~210 (main.py) + Frontend: PlanLibrary **387** + api **~150** = **~750** |
| **Изменённые файлы** | `backend/main.py`, `src/components/PlanLibrary.tsx`, `src/lib/api.ts` |
| **Артефакт на выходе** | `GET /api/plans` → список; `POST /api/plans` → id; `DELETE /api/plans/{id}`; UI списка планов |

---

## 15. Редактор плана (Plan Editor)

| Параметр | Значение |
|----------|----------|
| **Задача** | Рисование стен (линии) и прямоугольников на canvas. Сохранение в библиотеку. |
| **Строк кода** | **873** |
| **Изменённые файлы** | `src/components/PlanEditor.tsx` |
| **Артефакт на выходе** | `drawnPlan: { type, points }[]`; SVG preview; сохранение через API |

---

## 16. Чат поддержки (Support Chat)

| Параметр | Значение |
|----------|----------|
| **Задача** | UI чата с AI (backend /api/chat). Отправка сообщений, отображение ответов. |
| **Строк кода** | SupportChat: **909**, Support page: **132**, Backend chat: ~300 = **~1350** |
| **Изменённые файлы** | `src/components/SupportChat.tsx`, `src/pages/Support.tsx`, `backend/main.py` |
| **Артефакт на выходе** | UI чата; ответы AI в реальном времени |

---

## 17. Уведомления об ошибках в Telegram

| Параметр | Значение |
|----------|----------|
| **Задача** | При ошибках обработки видео — отправка в Telegram канал. |
| **Строк кода** | ~240 в main.py |
| **Изменённые файлы** | `backend/main.py` |
| **Артефакт на выходе** | Сообщение в Telegram с контекстом ошибки |

---

**Итого:** ~19 500+ строк кода, 20+ файлов.